# Probability & Statistics for Machine Learning: Interactive Laboratory

Machine learning models reason under uncertainty. Probability provides the axiomatic framework for quantifying uncertainty, while statistics provides the inferential tools for learning parameters from observed sample data, validating significance, and avoiding spurious correlations.

### Table of Contents
1. [Probability Distributions & Density Functions](#1-probability-distributions--density-functions)
2. [Bayesian Updating & The False Positive Paradox](#2-bayesian-updating--the-false-positive-paradox)
3. [Empirical Central Limit Theorem (CLT) Demonstration](#3-empirical-central-limit-theorem-clt-demonstration)
4. [Confidence Intervals: Analytical vs. Empirical Bootstrap](#4-confidence-intervals-analytical-vs-empirical-bootstrap)
5. [Hypothesis Testing & Permutation Tests (A/B Experimentation)](#5-hypothesis-testing--permutation-tests-ab-experimentation)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")
print("Loaded Probability and Statistics interactive lab.")

## 1. Probability Distributions & Density Functions
Visualizing Gaussian, Uniform, and Exponential distributions.

In [ ]:
x = np.linspace(-4, 6, 300)
gaussian_pdf = stats.norm.pdf(x, loc=1.0, scale=1.2)
exponential_pdf = stats.expon.pdf(x, scale=1.5)

plt.figure(figsize=(8, 4.5))
plt.plot(x, gaussian_pdf, label="Gaussian N(mu=1, sigma=1.2)", color="#1f77b4", lw=2)
plt.plot(x, exponential_pdf, label="Exponential (lambda=1/1.5)", color="#d62728", linestyle="--", lw=2)
plt.title("Probability Density Functions (PDF)", fontweight="bold")
plt.xlabel("x")
plt.ylabel("Density")
plt.legend()
plt.tight_layout()
plt.show()

## 2. Bayesian Updating & The False Positive Paradox
Bayes' Theorem: $P(D | +) = \frac{P(+ | D) P(D)}{P(+)}$.
When a disease or anomaly has low prevalence (prior $P(D) = 0.01$), even a test with 99% sensitivity and 95% specificity yields a surprising posterior.

In [ ]:
def compute_posterior(prior, sensitivity, false_positive_rate):
    evidence = (sensitivity * prior) + (false_positive_rate * (1.0 - prior))
    return (sensitivity * prior) / evidence

priors = np.logspace(-4, 0, 100)  # From 0.01% to 100%
posteriors = [compute_posterior(p, 0.99, 0.05) for p in priors]

plt.figure(figsize=(7, 4.5))
plt.plot(priors, posteriors, color="#2b5c8f", lw=2)
plt.xscale("log")
plt.axvline(0.01, color="red", linestyle=":", label="Prevalence = 1%")
plt.scatter([0.01], [compute_posterior(0.01, 0.99, 0.05)], color="red", zorder=5)
plt.title("Posterior Probability P(Condition | Positive Test) vs. Prior", fontweight="bold")
plt.xlabel("Prior Probability (Prevalence, Log Scale)")
plt.ylabel("Posterior P(Disease | Test = Positive)")
plt.legend()
plt.tight_layout()
plt.show()
print(f"Posterior at 1% prior: {compute_posterior(0.01, 0.99, 0.05):.2%}")

## 3. Empirical Central Limit Theorem (CLT) Demonstration
The Central Limit Theorem states that the distribution of normalized sample means approaches a normal distribution as sample size $n$ increases, regardless of the underlying population distribution.

In [ ]:
# Draw from highly non-Gaussian Exponential distribution
np.random.seed(42)
num_trials = 2000
sample_sizes = [2, 10, 50]

plt.figure(figsize=(12, 4))
for idx, n in enumerate(sample_sizes, 1):
    sample_means = [np.mean(np.random.exponential(scale=2.0, size=n)) for _ in range(num_trials)]
    plt.subplot(1, 3, idx)
    sns.histplot(sample_means, kde=True, color="#1f77b4", stat="density")
    plt.title(f"Sample Size n = {n}", fontweight="bold")
    plt.xlabel("Sample Mean")
plt.suptitle("Central Limit Theorem in Action (Exponential Population)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 4. Confidence Intervals: Analytical vs. Empirical Bootstrap
Estimating parameter uncertainty without assuming normality using the Bootstrap resampling method.

In [ ]:
# Skewed log-normal data
np.random.seed(123)
data = np.random.lognormal(mean=1.5, sigma=0.6, size=150)

# Method 1: Analytical Normal CI
mean_obs = np.mean(data)
se = stats.sem(data)
ci_analytic = stats.norm.interval(0.95, loc=mean_obs, scale=se)

# Method 2: Non-parametric Bootstrap CI
boot_means = [np.mean(np.random.choice(data, size=len(data), replace=True)) for _ in range(2500)]
ci_boot = (np.percentile(boot_means, 2.5), np.percentile(boot_means, 97.5))

print(f"Sample Mean:              {mean_obs:.3f}")
print(f"Analytical 95% CI:        [{ci_analytic[0]:.3f}, {ci_analytic[1]:.3f}]")
print(f"Bootstrap 95% CI:         [{ci_boot[0]:.3f}, {ci_boot[1]:.3f}]")

## 5. Hypothesis Testing & Permutation Tests (A/B Experimentation)
Evaluating whether an observed metric lift in an A/B test is statistically significant under the null hypothesis $H_0: \mu_A = \mu_B$.

In [ ]:
# Simulated CTR / Conversion Metric: Control vs Variant
np.random.seed(42)
control = np.random.normal(loc=0.10, scale=0.03, size=200)
variant = np.random.normal(loc=0.108, scale=0.03, size=200)

observed_diff = np.mean(variant) - np.mean(control)

# Welch's two-sample t-test
t_res = stats.ttest_ind(variant, control, equal_var=False)

# Permutation test (resampling under H0)
pooled = np.concatenate([control, variant])
n_ctrl = len(control)
perm_diffs = []
for _ in range(5000):
    shuffled = np.random.permutation(pooled)
    d = np.mean(shuffled[n_ctrl:]) - np.mean(shuffled[:n_ctrl])
    perm_diffs.append(d)

perm_p_val = np.mean(np.abs(perm_diffs) >= np.abs(observed_diff))

print(f"Observed Lift (Variant - Control): {observed_diff:.4f}")
print(f"Welch t-test p-value:              {t_res.pvalue:.4f}")
print(f"Permutation test p-value:          {perm_p_val:.4f}")
print(f"Statistical Decision (alpha=0.05): {'Statistically Significant' if perm_p_val < 0.05 else 'Inconclusive'}")